# 04 — Business impact and dashboard exports

This notebook turns technical outputs into something easier to show in a meeting.

## Outputs
- KPI summary
- asset-level summary
- cost and downtime estimates
- top-risk assets

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path('..')
DATA = BASE / 'data' / 'processed'
OUT = BASE / 'outputs'


In [ ]:
assets = pd.read_csv(DATA / 'asset_master.csv')
linked = pd.read_csv(DATA / 'pm_failure_linked.csv')
impact = pd.read_csv(DATA / 'business_impact.csv')
telemetry = pd.read_csv(DATA / 'telemetry_weekly.csv')

linked_nonnull = linked.dropna(subset=['days_to_next_failure']).copy()

kpis = {
    'asset_count': int(assets.shape[0]),
    'pm_event_count': int(pd.read_csv(DATA / 'pm_events.csv').shape[0]),
    'failure_event_count': int(pd.read_csv(DATA / 'failure_events.csv').shape[0]),
    'avg_days_to_next_failure': round(float(linked_nonnull['days_to_next_failure'].mean()), 2),
    'median_days_to_next_failure': round(float(linked_nonnull['days_to_next_failure'].median()), 2),
    'pct_ontime_pm': round(float(pd.read_csv(DATA / 'pm_events.csv')['ontime_flag'].mean() * 100), 2),
    'avg_failure_cost': round(float(pd.read_csv(DATA / 'failure_events.csv')['total_cost'].mean()), 2),
    'avg_downtime_hours': round(float(pd.read_csv(DATA / 'failure_events.csv')['downtime_hours'].mean()), 2),
}
pd.DataFrame({'metric': list(kpis.keys()), 'value': list(kpis.values())})

In [ ]:
top_risk = (
    telemetry.groupby('asset_id')
    .agg(
        avg_anomaly_score=('anomaly_score', 'mean'),
        fail_30d_rate=('failure_within_30d', 'mean'),
        max_alarm_count=('alarm_count', 'max')
    )
    .reset_index()
    .merge(assets[['asset_id', 'model', 'region', 'criticality']], on='asset_id', how='left')
    .sort_values(['fail_30d_rate', 'avg_anomaly_score'], ascending=[False, False])
)

top_risk.head(15)

In [ ]:
impact_summary = impact.agg({
    'downtime_hours': ['sum', 'mean'],
    'truck_roll_cost': ['sum', 'mean'],
    'repair_cost': ['sum', 'mean'],
    'estimated_revenue_loss': ['sum', 'mean'],
    'estimated_customers_impacted': ['sum', 'mean'],
})
impact_summary

In [ ]:
region_cost = (
    impact.merge(assets[['asset_id', 'region']], on='asset_id', how='left')
    .groupby('region')['estimated_revenue_loss']
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(region_cost.index, region_cost.values)
plt.title('Estimated revenue loss by region')
plt.ylabel('Revenue loss')
plt.tight_layout()
plt.show()

You can export these tables for Power BI, Tableau, Streamlit, or a React dashboard by writing them to CSV or JSON.